#Spark Learning

##By Knowing the process of read and write , we become a Data ingestion developer

connecting various sources (files or filesystem , db , dwh , api ,.. ) and loading data into storage env(data lake)

- csv
- json
- xml
- parquet
- ORC
- sql etc....


In [0]:
#To know Spark version

spark.version


## Few Facts about Unity Catalog

Dataobjects  is managed  in catalog with three namespace 


Catalog -> per domain or environemnt 

schema -> database 

tables , views , functions , volume 

Volume -  Non tabular data (files) , goverened access 

Catalog >> Schema  >> 
                    Table
                    View
                    functions
                    volume 

Volumes are used for managing Non tabular data (files)

In [0]:
%sql
create catalog if not exists izwd37dev;

create schema if not exists izwd37dev.wd37db;

--create volume if not exists izwd37dev.wd37db.rawdata;

create volume if not exists izwd37dev.wd37db.rawdatta;


## DBFS 

### Databricks File system 

distribuited virtual file system , linux posix format  runninng on top of your cloud storages 


/Volumes/catalog/schema/volume-name/path/to/file



dbfs:/ - uri   -> uniform resource identifier  (dbricks file system )

hdfs:/    -> hadoop distruibuited file system 

file:/     -> local file 

s3a:/   -> aws s3

gcs:/  -> google storage


adls:/   -> azure datalake 

In [0]:
%fs ls "dbfs:/Volumes/izwd37dev/wd37db/rawdata/"

In [0]:
%sql
drop volume izwd37dev.wd37db.rawdata

### Volumes operations using fs commands
-  List out files in Volumes ( Unity Catalog)

In [0]:
%fs ls "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/"

In [0]:
%fs ls "/Volumes/izwd37dev/wd37db/rawdatta/cust_sample.txt"

In [0]:
print(dbutils.fs.ls("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/"))

In [0]:
%fs mkdirs "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv"


In [0]:
%fs mkdirs "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json"
--dbutils.fs.mkdirs("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json")

what is dbfs:/ -> uri 
        hdfs:/ 
        file:/
        s3:/
        gcs:/

In [0]:
#read data from spark 
#spark SQL 

Spark Session

from pyspark.sql import SparkSession


spark = SparkSession.builder().getOrCreate()


In [0]:
print(spark)

create Dataframe from storage (files / dir ) To read the delimited data from any storage (dbfs , hdfs , lfs , cloud storages )

spark.read.csv opition -> dataframe

-- csv is the built in source

Loaded the custs file into rawdatta volume 

In [0]:
%fs ls "dbfs:/Volumes/izwd37dev/wd37db/rawdatta"

In [0]:
custdata = spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs") #file path
print(type(custdata)) #dataframe
custdata.show() #show is action, similar to collect 
# show action , display default 20 records 

In [0]:
#Describe table
custdata.printSchema()

In [0]:
#view few more records
custdata.show(10,False)
#custdata.show(200)                                   

In [0]:
#Supported in Databricks to show result in formatted wy to filter, ascend, descend data 
display(custdata)

In [0]:
#custdata is a dataframe and Schema provides schema format in spark Structtype and Structfiled type 
print(custdata.schema)

In [0]:
#Changing default column name _c0,_c1 etc to actual custid, fname etc
custdata= spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs").toDF("custid","fname","lname","age","profession")

custdata.show(5,False)  #Display only 5 records
custdata.printSchema()

### suppose we recivied a file with header
### column names we need to pick from the header

In [0]:
#Without header option
custdata = spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header")

custdata.show(5)

custdata.printSchema()

In [0]:
%python
# how to get the number of records in the dataframe
# select count(1) from table 
# count is an action , its return integer result , trigger execution , job is created
custdata.count()

overriding the defaults with option

enable the header

In [0]:
#With header option
custdata = spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header",header=True)

custdata.show(5)

custdata.printSchema()

print(f"Record count in custs is {custdata.count()}")

In [0]:
# data type for all columns treated default as string
# based on the data we have generate the schema with proper data type 
# performance if we read large data inferSchema is not a good option 
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header",header=True,inferSchema=True)

custdata.show(5)

custdata.printSchema()


## Different delimited (|)

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/emp1.csv

In [0]:
emp_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/emp1.csv",header=True,inferSchema=True,sep="|")
emp_df.show(5)

emp_df.printSchema()

spark.read.csv()

-> required input **path**

-> path could be a file or dir , list of dir ...